# Latent AutoRegression (LAR) Training

Fine-tunes gpt2-medium so that the middle transformer layers (layer_a through layer_b) learn to autoregressively predict the next latent representation, enabling inference-time generation that bypasses the token encoder.

**Zones:**
- **Encoder**: `transformer.h[0 .. layer_a-1]` — tokens → latent
- **Latent AR core**: `transformer.h[layer_a .. layer_b-1]` — latent → latent
- **Decoder**: `transformer.h[layer_b .. end]` — latent → logits

**Penalty**: mean squared L2 distance between `h_b[:, i, :]` (LAR output at position i) and `h_a[:, i+1, :]` (encoder output at position i+1). Trains the LAR core to step forward one token in latent space.

---
**Colab setup:** Before running, ensure the following files are accessible (e.g. via Google Drive mount):
- `wiki_tokens_train.dat` + `wiki_tokens_train_meta.npy` — training corpus
- `latent_ar_checkpoint.pt` — (optional) resume from a previous run

Set `DATA_DIR` and `CKPT_PATH` in the Config cell to point to these files.

## 1. Environment Setup

In [ ]:
import os, sys

try:
    from google.colab import drive
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    drive.mount('/content/drive')

    if not os.path.exists('/content/minGPT-LAR'):
        !git clone https://github.com/JCRaymond/minGPT-LAR /content/minGPT-LAR
    os.chdir('/content/minGPT-LAR')

    %pip install -e . -q

repo_root = os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f'Running on: {"Colab" if IS_COLAB else "local"}')
print(f'Repo root:  {repo_root}')

## 2. Imports

In [9]:
import time
from types import SimpleNamespace

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint as grad_ckpt

from mingpt.model import GPT
from mingpt.bpe import BPETokenizer
from mingpt.utils import set_seed

## 3. Config

Adjust `DATA_DIR` and `CKPT_PATH` to point to your data files.
On Colab with Drive mounted, these will typically be under `/content/drive/MyDrive/`.

In [ ]:
# --- Paths ---
DRIVE_DIR = '/content/drive/MyDrive/latent_ar'
_PROJECT_DIR = os.path.join(repo_root, 'projects', 'latent_ar')

DATA_DIR  = DRIVE_DIR if IS_COLAB else _PROJECT_DIR
CKPT_PATH = os.path.join(DRIVE_DIR if IS_COLAB else _PROJECT_DIR, 'latent_ar_checkpoint.pt')

# --- Device ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
elif device == 'cpu':
    torch.set_num_threads(10)

# --- Model ---
model_type = 'gpt2-medium'   # 24 layers, 345M params

# --- LAR split points ---
layer_a = 6    # last encoder block (1-indexed)
layer_b = 18   # last LAR core block (1-indexed)

# --- Training ---
lambda_penalty = 4e-4
block_size     = 1024
batch_size     = 4 if device == 'cuda' else 2
learning_rate  = 3e-5
weight_decay   = 0.1
betas          = (0.9, 0.95)
max_iters      = 1000
epochs         = 4
grad_norm_clip = 1.0
log_interval   = 10
ckpt_interval  = 100

print(f'Data dir:   {DATA_DIR}')
print(f'Checkpoint: {CKPT_PATH}')
print(f'batch_size: {batch_size}, max_iters: {max_iters}, epochs: {epochs}')

## 4. Cache Base Weights to Drive

Colab's local storage is wiped between sessions. This cell downloads the base gpt2-medium weights once and saves them to Drive so subsequent sessions don't need to re-download from HuggingFace.

In [12]:
BASE_WEIGHTS_PATH = os.path.join(DRIVE_DIR if IS_COLAB else _PROJECT_DIR, 'gpt2-medium-base.pt')

if os.path.exists(BASE_WEIGHTS_PATH):
    print(f'Base weights already cached at {BASE_WEIGHTS_PATH}')
else:
    print('Downloading gpt2-medium base weights from HuggingFace...')
    _base = GPT.from_pretrained(model_type)
    torch.save(_base.state_dict(), BASE_WEIGHTS_PATH)
    del _base
    print(f'Base weights saved to {BASE_WEIGHTS_PATH}')

number of parameters: 354.82M


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


AssertionError: 

## 4. Data Loading

In [ ]:
TRAIN_PATH      = os.path.join(DATA_DIR, 'wiki_tokens_train.dat')
TRAIN_META_PATH = os.path.join(DATA_DIR, 'wiki_tokens_train_meta.npy')

if not (os.path.exists(TRAIN_PATH) and os.path.exists(TRAIN_META_PATH)):
    raise FileNotFoundError(
        f'Training data not found at {TRAIN_PATH}.\n'
        'Run wiki_data.py locally to build and split the corpus, '
        'then transfer wiki_tokens_train.dat and wiki_tokens_train_meta.npy to DATA_DIR.'
    )

n_tokens = int(np.load(TRAIN_META_PATH)[0])
tokens   = np.memmap(TRAIN_PATH, dtype=np.int32, mode='r', shape=(n_tokens,))
print(f'Loaded {n_tokens:,} train tokens')

## 5. Dataset

In [ ]:
class TokenDataset(Dataset):
    """
    Randomly samples block_size-length chunks from a memory-mapped token array.
    Uses a per-instance system-entropy RNG so samples differ between runs.
    __len__ is a fixed epoch size to avoid DataLoader building a full index
    permutation over the entire corpus (~38 GB for Wikipedia).
    """
    def __init__(self, tokens, block_size, epoch_size):
        self.tokens     = tokens
        self.block_size = block_size
        self.max_start  = len(tokens) - block_size - 1
        self.epoch_size = epoch_size
        self.rng        = np.random.default_rng()  # system-entropy seed

    def __len__(self):
        return self.epoch_size

    def __getitem__(self, idx):
        pos   = int(self.rng.integers(0, self.max_start))
        chunk = torch.from_numpy(
            self.tokens[pos : pos + self.block_size + 1].astype(np.int64)
        )
        return chunk[:-1], chunk[1:]


epoch_size = max_iters * batch_size
dataset    = TokenDataset(tokens, block_size, epoch_size)
loader     = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
print(f'epoch_size={epoch_size:,}, block_size={block_size}')

## 6. Model Loading

In [ ]:
def make_hook(buf):
    def hook(module, input, output):
        buf['act'] = output
    return hook


model = GPT.from_pretrained(model_type)
model.load_state_dict(torch.load(BASE_WEIGHTS_PATH, map_location=device, weights_only=True))

if os.path.exists(CKPT_PATH):
    print(f'Resuming from LAR checkpoint: {CKPT_PATH}')
    model.load_state_dict(torch.load(CKPT_PATH, map_location=device, weights_only=True))
else:
    print('No LAR checkpoint found — starting from base pretrained weights.')

model.to(device)
model.train()

if device == 'cuda':
    torch.cuda.empty_cache()

n_params = sum(p.numel() for p in model.parameters())
n_layers = len(model.transformer.h)
print(f'Model: {model_type}, {n_params/1e6:.1f}M params, {n_layers} transformer blocks')
assert layer_b <= n_layers, f'layer_b={layer_b} exceeds model depth {n_layers}'

## 7. Training Setup

Gradient checkpointing recomputes activations during backward instead of storing them, reducing activation memory ~40× at the cost of a second forward pass per block. Essential for fitting gpt2-medium on a single GPU.

In [ ]:
# Gradient checkpointing — guard against applying twice if cell is re-run
if not getattr(model, '_grad_ckpt_applied', False):
    for block in model.transformer.h:
        orig_fwd = block.forward
        block.forward = lambda x, f=orig_fwd: grad_ckpt(f, x, use_reentrant=False)
    model._grad_ckpt_applied = True

# Remove any existing hooks before re-registering
try:
    handle_a.remove()
    handle_b.remove()
except:
    pass

h_a_buf, h_b_buf = {}, {}
handle_a = model.transformer.h[layer_a - 1].register_forward_hook(make_hook(h_a_buf))
handle_b = model.transformer.h[layer_b - 1].register_forward_hook(make_hook(h_b_buf))
print(f'Hooks on block {layer_a-1} (encoder out) and block {layer_b-1} (LAR out)')

# Optimizer
train_cfg = SimpleNamespace(learning_rate=learning_rate, weight_decay=weight_decay, betas=betas)
optimizer = model.configure_optimizers(train_cfg)

Hooks on block 5 (encoder out) and block 17 (LAR out)


## 8. Training Loop

In [ ]:
data_iter = iter(loader)
t0        = time.time()

for epoch in range(1, epochs + 1):
    print(f'Epoch: {epoch}')
    ep_ckpt_path = os.path.join(DRIVE_DIR if IS_COLAB else _PROJECT_DIR, f'latent_ar_checkpoint-{epoch}.pt')

    for iter_num in range(1, max_iters + 1):
        try:
            x, y = next(data_iter)
        except StopIteration:
            data_iter = iter(loader)
            x, y = next(data_iter)

        x, y = x.to(device), y.to(device)

        logits, ce_loss = model(x, y)

        h_a     = h_a_buf['act']
        h_b     = h_b_buf['act']
        penalty = ((h_a[:, 1:, :] - h_b[:, :-1, :]) ** 2).sum(dim=-1).mean()

        loss = ce_loss + lambda_penalty * penalty

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_norm_clip)
        optimizer.step()

        if iter_num % log_interval == 0:
            t1          = time.time()
            ms_per_iter = (t1 - t0) / log_interval * 1000
            t0          = t1
            penalty_contrib = lambda_penalty * penalty.item()
            ratio       = penalty_contrib / (ce_loss.item() + 1e-8)
            ratio_str   = f' [penalty/ce: {ratio:.2f}]'
            if ratio > 10:
                ratio_str += ' <- consider reducing lambda_penalty'
            elif ratio < 0.01:
                ratio_str += ' <- consider increasing lambda_penalty'
            print(
                f'iter {iter_num:6d} | '
                f'loss {loss.item():.4f} | '
                f'ce {ce_loss.item():.4f} | '
                f'penalty {penalty.item():.4f} | '
                f'{ms_per_iter:.0f}ms/iter'
                + ratio_str
            )

        if iter_num % ckpt_interval == 0:
            torch.save(model.state_dict(), ep_ckpt_path)
            print(f'Checkpoint saved -> {ep_ckpt_path}')

    torch.save(model.state_dict(), ep_ckpt_path)
    print(f'Epoch {epoch} complete. Checkpoint: {ep_ckpt_path}')

## 9. Evaluate on Training Data

Measures mean CE loss and LAR penalty over a random sample of the training corpus.

In [34]:
eval_batches = 50
rng = np.random.default_rng()

model.eval()
ce_losses, penalties, l2_distances = [], [], []

with torch.no_grad():
    for _ in range(eval_batches):
        starts = rng.integers(0, len(tokens) - block_size - 1, size=batch_size)
        x_eval = torch.from_numpy(
            np.stack([tokens[s : s + block_size + 1].astype(np.int64) for s in starts])
        )
        x_in, y_in = x_eval[:, :-1].to(device), x_eval[:, 1:].to(device)

        _, ce = model(x_in, y_in)
        h_a = h_a_buf['act']
        h_b = h_b_buf['act']
        sq_dists = ((h_a[:, 1:, :] - h_b[:, :-1, :]) ** 2).sum(dim=-1)  # (B, T-1)

        ce_losses.append(ce.item())
        penalties.append(sq_dists.mean().item())
        l2_distances.append(sq_dists.sqrt().mean().item())

model.train()

print(f'Eval over {eval_batches} batches ({eval_batches * batch_size * block_size:,} tokens):')
print(f'  Mean CE loss:     {np.mean(ce_losses):.4f}')
print(f'  Mean penalty:     {np.mean(penalties):.4f}')
print(f'  Mean L2 distance: {np.mean(l2_distances):.4f}')

Eval over 50 batches (204,800 tokens):
  Mean CE loss:     3.0302
  Mean penalty:     1761.6724
  Mean L2 distance: 40.8424


## 9. Generate Sample

Runs standard token-by-token generation to verify language modelling ability is preserved after LAR fine-tuning.

In [ ]:
gen_model = GPT.from_pretrained(model_type)
gen_model.load_state_dict(torch.load(CKPT_PATH, map_location=device, weights_only=True))
gen_model.to(device)
gen_model.eval()

tokenizer = BPETokenizer()
prompt    = 'Wikipedia is a free online encyclopedia'
x         = tokenizer(prompt).to(device)

with torch.no_grad():
    y = gen_model.generate(x, max_new_tokens=100, do_sample=True, top_k=40)

print(tokenizer.decode(y[0]))